# 04 – Preparazione sperimentale

Prepariamo i dati del dataset BRIGHT necessari
per gli esperimenti di addestramento e valutazione.

Definiamo gli split operativi, selezioniamo un
campione riproducibile, verifichiamo la qualità
dei dati e costruiamo il Dataset multimodale.

Le informazioni prodotte saranno utilizzate
nei successivi notebook sperimentali.

In [1]:
from pathlib import Path

import pandas as pd
import torch

ROOT = Path.cwd()

print("Directory del progetto:", ROOT)

Directory del progetto: /Users/antonyogrossi/tesi/riproduzione_dfc2025


## Preparazione degli split operativi

Per estendere l'addestramento oltre il test su una
singola patch, costruiamo liste operative a partire
dalle suddivisioni ufficiali del DFC2025 Track 2.

Uniamo train e holdout, eliminiamo gli identificativi
duplicati ed escludiamo dal training quelli presenti
in validation o test.

Le liste ufficiali rimangono inalterate.
La separazione per identificativo non garantisce
l'assenza di sovrapposizioni geografiche.

In [2]:
from bright_reproduction.splits import SuddivisioniBright

CARTELLA_SPLIT = (
    ROOT / "BRIGHT"
    / "bda_benchmark"
    / "dataset"
    / "splitname"
    / "dfc25"
)

suddivisioni = SuddivisioniBright.da_cartella(
    CARTELLA_SPLIT
)

split_operativi = suddivisioni.liste_operative()

for nome, lista in split_operativi.items():
    print(f"{nome}: {len(lista)} esempi")

train: 2888 esempi
val: 349 esempi
test: 156 esempi


In [3]:
CARTELLA_SPLIT_OPERATIVI = (
    ROOT / "risultati" / "04_baseline_unet" / "splits"
)

destinazione = suddivisioni.salva_liste_operative(
    CARTELLA_SPLIT_OPERATIVI
)

for file in sorted(destinazione.glob("*_operativo.txt")):
    righe = file.read_text(encoding="utf-8").splitlines()
    print(f"{file.name}: {len(righe)} identificativi")

test_operativo.txt: 156 identificativi
train_operativo.txt: 2888 identificativi
val_operativo.txt: 349 identificativi


## Selezione del campione operativo

Prima di estendere l'addestramento all'intero
dataset, selezioniamo un campione ridotto di
training e validation.

Analizziamo la distribuzione degli esempi per
evento, in modo da evitare una selezione
limitata a una sola tipologia di disastro.

Il campione verrà utilizzato per verificare
la pipeline di addestramento e valutazione.

In [4]:
display(suddivisioni.eventi())

,train,holdout,val,test
bata-explosion,70,23,14,0
beirut-explosion,89,27,17,0
congo-volcano,81,27,15,0
haiti-earthquake,48,15,10,0
hawaii-wildfire,42,14,9,0
la_palma-volcano,647,189,97,0
libya-flood,83,26,15,0
marshall-wildfire,0,0,0,77
morocco-earthquake,392,116,59,0
noto-earthquake,0,0,0,79


In [5]:
from bright_reproduction.splits import SuddivisioniBright

suddivisioni = SuddivisioniBright.da_cartella(
    CARTELLA_SPLIT
)

campione_operativo = suddivisioni.campione_per_evento(
    n_train=2,
    n_val=1,
    seed=42
)

for nome, lista in campione_operativo.items():

    print(f"\n{nome.upper()}: {len(lista)} esempi")

    for id_ in lista:
        print(id_)


TRAIN: 18 esempi
bata-explosion_00000096
bata-explosion_00000016
beirut-explosion_00000005
beirut-explosion_00000110
congo-volcano_00000041
congo-volcano_00000037
haiti-earthquake_00000015
haiti-earthquake_00000009
hawaii-wildfire_00000054
hawaii-wildfire_00000010
la_palma-volcano_00000769
la_palma-volcano_00000849
libya-flood_00000081
libya-flood_00000013
morocco-earthquake_00000340
morocco-earthquake_00000240
turkey-earthquake_00000036
turkey-earthquake_00000034

VAL: 9 esempi
bata-explosion_00000012
beirut-explosion_00000047
congo-volcano_00000014
haiti-earthquake_00000059
hawaii-wildfire_00000002
la_palma-volcano_00000691
libya-flood_00000017
morocco-earthquake_00000448
turkey-earthquake_00000845


## Download del campione operativo

Scarichiamo 18 esempi di training e 9 esempi
di validation, selezionati in maniera riproducibile
e bilanciata per evento.

Le immagini e le maschere dei due split vengono
salvate in cartelle distinte.

I dati del test rimangono esclusi dall'esperimento.

In [6]:
from bright_reproduction.download import (
    scarica_campione_operativo
)

BASE_URL = (
    "https://huggingface.co/datasets/"
    "Kullervo/BRIGHT/resolve/main/"
)

URL_DATI = BASE_URL + "dfc25_track2_trainval.zip"
URL_LABELS = BASE_URL + "dfc25_track2_val_labels.zip"

CARTELLA_BASELINE = ROOT / "dati" / "bright_baseline"

In [7]:
risultati_download = scarica_campione_operativo(
    URL_DATI,
    URL_LABELS,
    campione_operativo,
    CARTELLA_BASELINE
)

for split, risultati in risultati_download.items():

    print(f"\n{split.upper()}")
    print("File verificati:", len(risultati))

    display(
        pd.DataFrame(risultati)["stato"].value_counts()
    )


TRAIN
File verificati: 54


stato
già presente e verificato    54
Name: count, dtype: int64


VAL
File verificati: 27


stato
già presente e verificato    27
Name: count, dtype: int64

## Controllo del campione operativo

Verifichiamo la disponibilità dei file scaricati,
la distribuzione delle classi e la percentuale
di pixel appartenenti agli edifici.

Training e validation vengono analizzati
separatamente, riutilizzando le funzionalità
del package bright_reproduction.

In [8]:
from bright_reproduction.campione import (
    controlla_disponibilita,
    statistiche_classi,
    statistiche_edifici
)

for split, identificativi in campione_operativo.items():

    disponibilita, riepilogo = controlla_disponibilita(
        CARTELLA_BASELINE / split,
        {split: identificativi}
    )

    print(f"\n{split.upper()}")
    display(riepilogo.loc[[split]].astype(int))

    assert disponibilita["completo"].all()

print("Tutti gli esempi sono completi.")


TRAIN


,esempi_in_lista,ottici_presenti,sar_presenti,maschere_presenti,esempi_completi
split,,,,,
train,18,18,18,18,18



VAL


,esempi_in_lista,ottici_presenti,sar_presenti,maschere_presenti,esempi_completi
split,,,,,
val,9,9,9,9,9


Tutti gli esempi sono completi.


In [9]:
statistiche_campione = {}

for split, identificativi in campione_operativo.items():

    statistiche, riepilogo, totale = statistiche_classi(
        CARTELLA_BASELINE / split,
        identificativi
    )

    statistiche_campione[split] = {
        "per_esempio": statistiche,
        "riepilogo": riepilogo,
        "totale_pixel": totale
    }

    print(f"\n{split.upper()}")
    print(f"Pixel validi: {totale:,}")

    display(riepilogo.round(2))


TRAIN
Pixel validi: 18,874,368


,pixel,percentuale
Sfondo,16342136,86.58
Edificio integro,2113342,11.20
Edificio danneggiato,338140,1.79
Edificio distrutto,80750,0.43



VAL
Pixel validi: 9,437,184


,pixel,percentuale
Sfondo,8023499,85.02
Edificio integro,1307077,13.85
Edificio danneggiato,33608,0.36
Edificio distrutto,73000,0.77


In [10]:
for split, risultati in statistiche_campione.items():

    print(f"\n{split.upper()}")

    tabella_edifici = statistiche_edifici(
        risultati["per_esempio"]
    )

    display(tabella_edifici.round(2))


TRAIN


,pixel_sfondo,pixel_edificio,percentuale_edificio
id,,,
bata-explosion_00000096,1048576,0,0.00
bata-explosion_00000016,902043,146533,13.97
beirut-explosion_00000005,1005559,43017,4.10
beirut-explosion_00000110,500331,548245,52.28
congo-volcano_00000041,1038899,9677,0.92
congo-volcano_00000037,962967,85609,8.16
haiti-earthquake_00000015,678766,369810,35.27
haiti-earthquake_00000009,1008888,39688,3.78
hawaii-wildfire_00000054,1026161,22415,2.14



VAL


,pixel_sfondo,pixel_edificio,percentuale_edificio
id,,,
bata-explosion_00000012,1035436,13140,1.25
beirut-explosion_00000047,499778,548798,52.34
congo-volcano_00000014,1047804,772,0.07
haiti-earthquake_00000059,630849,417727,39.84
hawaii-wildfire_00000002,921897,126679,12.08
la_palma-volcano_00000691,1048576,0,0.00
libya-flood_00000017,1031515,17061,1.63
morocco-earthquake_00000448,1048576,0,0.00
turkey-earthquake_00000845,759068,289508,27.61


## Verifica delle griglie geografiche

Controlliamo la compatibilità delle griglie
delle immagini ottiche, SAR e delle maschere
per tutti gli esempi del campione operativo.

La coincidenza delle griglie dichiarate è una
condizione necessaria, ma non garantisce la
perfetta registrazione del contenuto delle immagini.

Il controllo rigoroso con tolleranza di 1e-6 pixel
ha evidenziato una differenza nella risoluzione
spaziale dichiarata del SAR nei tre esempi Hawaii.

Lo scarto massimo osservato è di circa 0,022558
pixel. Per la baseline preliminare adottiamo una
tolleranza operativa di 0,05 pixel, senza
ricampionare o modificare i raster originali.

In [11]:
from bright_reproduction.campione import (
    controlla_griglie_campione
)

In [12]:
TOLLERANZA_GRIGLIE = 0.05

risultati_griglie = {}

print("Tolleranza utilizzata:", TOLLERANZA_GRIGLIE)

for split, identificativi in campione_operativo.items():

    tabella = controlla_griglie_campione(
        CARTELLA_BASELINE / split,
        identificativi,
        tolleranza=TOLLERANZA_GRIGLIE
    )

    risultati_griglie[split] = tabella

    print(f"\n{split.upper()}")
    print("Esempi controllati:", len(tabella))
    print("Griglie compatibili:", tabella["compatibile"].sum())

    display(tabella)

assert all(
    tabella["compatibile"].all()
    for tabella in risultati_griglie.values()
), "Sono presenti griglie incompatibili."

print("Tutte le griglie sono compatibili.")

Tolleranza utilizzata: 0.05

TRAIN
Esempi controllati: 18
Griglie compatibili: 18


,compatibile,scarto_massimo_pixel,errore
id,,,
bata-explosion_00000096,True,0.000000e+00,None
bata-explosion_00000016,True,0.000000e+00,None
beirut-explosion_00000005,True,0.000000e+00,None
beirut-explosion_00000110,True,0.000000e+00,None
congo-volcano_00000041,True,0.000000e+00,None
congo-volcano_00000037,True,0.000000e+00,None
haiti-earthquake_00000015,True,0.000000e+00,None
haiti-earthquake_00000009,True,1.862645e-09,None
hawaii-wildfire_00000054,True,2.255780e-02,None



VAL
Esempi controllati: 9
Griglie compatibili: 9


,compatibile,scarto_massimo_pixel,errore
id,,,
bata-explosion_00000012,True,0.000000e+00,None
beirut-explosion_00000047,True,0.000000e+00,None
congo-volcano_00000014,True,0.000000e+00,None
haiti-earthquake_00000059,True,1.862645e-09,None
hawaii-wildfire_00000002,True,2.255780e-02,None
la_palma-volcano_00000691,True,0.000000e+00,None
libya-flood_00000017,True,2.328306e-10,None
morocco-earthquake_00000448,True,2.328306e-10,None
turkey-earthquake_00000845,True,0.000000e+00,None


Tutte le griglie sono compatibili.


In [13]:
import bright_reproduction.geospatial as geo

for split, tabella in risultati_griglie.items():

    incompatibili = tabella.index[
        tabella["scarto_massimo_pixel"] > 1e-6
    ]

    for id_ in incompatibili:

        cartella = CARTELLA_BASELINE / split

        ottico = (
            cartella / "pre-event"
            / f"{id_}_pre_disaster.tif"
        )

        for categoria, suffisso in [
            ("post-event", "_post_disaster.tif"),
            ("target", "_building_damage.tif")
        ]:

            percorso = (
                cartella / categoria
                / f"{id_}{suffisso}"
            )

            risultato = geo.confronta_trasformazioni(
                ottico, percorso
            )

            print(f"\n{split.upper()} - {id_}")
            print("Categoria:", categoria)

            for chiave, valore in risultato.items():
                if chiave != "scarti":
                    print(chiave, ":", valore)

            display(pd.DataFrame(risultato["scarti"]))


TRAIN - hawaii-wildfire_00000054
Categoria: post-event
risoluzione_riferimento : (0.3768599696565629, 0.3768671526667204)
risoluzione_raster : (0.37686827154079405, 0.3768682660537119)
trasformazione_riferimento : (0.3768599696565629, 0.0, -17440313.358379304, 0.0, -0.3768671526667204, 2376111.467437854, 0.0, 0.0, 1.0)
trasformazione_raster : (0.37686827154079405, 0.0, -17440313.358379304, 0.0, -0.3768682660537119, 2376111.467437854, 0.0, 0.0, 1.0)


,angolo,delta_colonna,delta_riga
0,"(0, 0)",0.000000,0.000000
1,"(1024, 0)",0.022558,0.000000
2,"(0, 1024)",0.000000,0.003025
3,"(1024, 1024)",0.022558,0.003025



TRAIN - hawaii-wildfire_00000054
Categoria: target
risoluzione_riferimento : (0.3768599696565629, 0.3768671526667204)
risoluzione_raster : (0.37685996965592494, 0.3768671526668186)
trasformazione_riferimento : (0.3768599696565629, 0.0, -17440313.358379304, 0.0, -0.3768671526667204, 2376111.467437854, 0.0, 0.0, 1.0)
trasformazione_raster : (0.37685996965592494, 0.0, -17440313.358379304, 0.0, -0.3768671526668186, 2376111.467437854, 0.0, 0.0, 1.0)


,angolo,delta_colonna,delta_riga
0,"(0, 0)",0.0,0.0
1,"(1024, 0)",0.0,0.0
2,"(0, 1024)",0.0,0.0
3,"(1024, 1024)",0.0,0.0



TRAIN - hawaii-wildfire_00000010
Categoria: post-event
risoluzione_riferimento : (0.3768599696565629, 0.3768671526667204)
risoluzione_raster : (0.37686827154079405, 0.3768682660537119)
trasformazione_riferimento : (0.3768599696565629, 0.0, -17439541.53215919, 0.0, -0.3768671526667204, 2379198.7722733663, 0.0, 0.0, 1.0)
trasformazione_raster : (0.37686827154079405, 0.0, -17439541.53215919, 0.0, -0.3768682660537119, 2379198.772273366, 0.0, 0.0, 1.0)


,angolo,delta_colonna,delta_riga
0,"(0, 0)",0.000000,9.313226e-10
1,"(1024, 0)",0.022558,9.313226e-10
2,"(0, 1024)",0.000000,3.025227e-03
3,"(1024, 1024)",0.022558,3.025227e-03


... (2 outputs left out)


VAL - hawaii-wildfire_00000002
Categoria: post-event
risoluzione_riferimento : (0.3768599696565629, 0.3768671526667204)
risoluzione_raster : (0.37686827154079405, 0.3768682660537119)
trasformazione_riferimento : (0.3768599696565629, 0.0, -17442242.923929594, 0.0, -0.3768671526667204, 2379970.5984822437, 0.0, 0.0, 1.0)
trasformazione_raster : (0.37686827154079405, 0.0, -17442242.923929594, 0.0, -0.3768682660537119, 2379970.5984822437, 0.0, 0.0, 1.0)


,angolo,delta_colonna,delta_riga
0,"(0, 0)",0.000000,0.000000
1,"(1024, 0)",0.022558,0.000000
2,"(0, 1024)",0.000000,0.003025
3,"(1024, 1024)",0.022558,0.003025



VAL - hawaii-wildfire_00000002
Categoria: target
risoluzione_riferimento : (0.3768599696565629, 0.3768671526667204)
risoluzione_raster : (0.37685996965592494, 0.3768671526668186)
trasformazione_riferimento : (0.3768599696565629, 0.0, -17442242.923929594, 0.0, -0.3768671526667204, 2379970.5984822437, 0.0, 0.0, 1.0)
trasformazione_raster : (0.37685996965592494, 0.0, -17442242.923929594, 0.0, -0.3768671526668186, 2379970.5984822437, 0.0, 0.0, 1.0)


,angolo,delta_colonna,delta_riga
0,"(0, 0)",0.000000e+00,0.0
1,"(1024, 0)",-7.450581e-09,0.0
2,"(0, 1024)",0.000000e+00,0.0
3,"(1024, 1024)",-7.450581e-09,0.0


## Controllo delle sovrapposizioni geografiche

Verifichiamo se gli esempi di training e validation
condividono porzioni di territorio, confrontando
le estensioni geografiche delle immagini ottiche.

Il controllo utilizza rettangoli di delimitazione
in un sistema di coordinate comune ed è una
verifica preliminare del rischio di data leakage
geografico.

In [14]:
from bright_reproduction.campione import (
    controlla_sovrapposizioni
)

sovrapposizioni_geo = controlla_sovrapposizioni(
    CARTELLA_BASELINE,
    campione_operativo
)

numero_coppie = (
    len(campione_operativo["train"])
    * len(campione_operativo["val"])
)

print("Coppie confrontate:", numero_coppie)
print("Sovrapposizioni rilevate:", len(sovrapposizioni_geo))

display(sovrapposizioni_geo)

Coppie confrontate: 162
Sovrapposizioni rilevate: 0


,id_train,id_val


## Validità dei raster e controllo dei NoData

Analizziamo le dimensioni, il numero di bande,
i tipi di dato e la presenza di valori non validi
nelle immagini ottiche, SAR e nelle maschere.

Questi controlli permettono di definire il
preprocessing necessario per l'addestramento.

In [15]:
from bright_reproduction.campione import (
    controlla_raster_campione
)

risultati_raster = {}

for split, identificativi in campione_operativo.items():

    tabella = controlla_raster_campione(
        CARTELLA_BASELINE / split,
        identificativi
    )

    risultati_raster[split] = tabella

    print(f"\n{split.upper()}")
    print("Raster controllati:", len(tabella))

    display(
        tabella.groupby("categoria").agg(
            file=("id", "size"),
            bande=("bande", "unique"),
            altezza=("altezza", "unique"),
            larghezza=("larghezza", "unique"),
            dtype=("dtype", "unique"),
            valori_invalidi=("totale_valori_invalidi", "sum")
        )
    )


TRAIN
Raster controllati: 54


,file,bande,altezza,larghezza,dtype,valori_invalidi
categoria,,,,,,
post-event,18,[1],[1024],[1024],"[('uint8',)]",2987625
pre-event,18,[3],[1024],[1024],"[('uint8', 'uint8', 'uint8')]",5965509
target,18,[1],[1024],[1024],"[('uint8',)]",0



VAL
Raster controllati: 27


,file,bande,altezza,larghezza,dtype,valori_invalidi
categoria,,,,,,
post-event,9,[1],[1024],[1024],"[('uint8',)]",1315658
pre-event,9,[3],[1024],[1024],"[('uint8', 'uint8', 'uint8')]",3741212
target,9,[1],[1024],[1024],"[('uint8',)]",0


In [16]:
for split, tabella in risultati_raster.items():

    problematici = tabella[
        tabella["totale_valori_invalidi"] > 0
    ]

    print(f"\n{split.upper()}")
    print("Raster con valori non validi:", len(problematici))

    display(problematici)


TRAIN
Raster con valori non validi: 25


,id,categoria,bande,altezza,larghezza,dtype,nodata,invalidi_per_banda,totale_valori_invalidi
0,bata-explosion_00000096,pre-event,3,1024,1024,"('uint8', 'uint8', 'uint8')",0.0,"[969183, 969183, 969183]",2907549
1,bata-explosion_00000096,post-event,1,1024,1024,"('uint8',)",0.0,[990223],990223
3,bata-explosion_00000016,pre-event,3,1024,1024,"('uint8', 'uint8', 'uint8')",0.0,"[1314, 0, 9]",1323
4,bata-explosion_00000016,post-event,1,1024,1024,"('uint8',)",0.0,[56660],56660
6,beirut-explosion_00000005,pre-event,3,1024,1024,"('uint8', 'uint8', 'uint8')",0.0,"[20259, 4752, 25231]",50242
9,beirut-explosion_00000110,pre-event,3,1024,1024,"('uint8', 'uint8', 'uint8')",0.0,"[23106, 652, 12605]",36363
10,beirut-explosion_00000110,post-event,1,1024,1024,"('uint8',)",0.0,[611110],611110
12,congo-volcano_00000041,pre-event,3,1024,1024,"('uint8', 'uint8', 'uint8')",0.0,"[3, 1, 18]",22
13,congo-volcano_00000041,post-event,1,1024,1024,"('uint8',)",0.0,[46102],46102
15,congo-volcano_00000037,pre-event,3,1024,1024,"('uint8', 'uint8', 'uint8')",0.0,"[5883, 5885, 5954]",17722



VAL
Raster con valori non validi: 13


,id,categoria,bande,altezza,larghezza,dtype,nodata,invalidi_per_banda,totale_valori_invalidi
0,bata-explosion_00000012,pre-event,3,1024,1024,"('uint8', 'uint8', 'uint8')",0.0,"[967255, 967255, 967255]",2901765
1,bata-explosion_00000012,post-event,1,1024,1024,"('uint8',)",0.0,[974171],974171
3,beirut-explosion_00000047,pre-event,3,1024,1024,"('uint8', 'uint8', 'uint8')",0.0,"[36466, 518, 15062]",52046
6,congo-volcano_00000014,pre-event,3,1024,1024,"('uint8', 'uint8', 'uint8')",0.0,"[0, 0, 2]",2
7,congo-volcano_00000014,post-event,1,1024,1024,"('uint8',)",0.0,[35698],35698
9,haiti-earthquake_00000059,pre-event,3,1024,1024,"('uint8', 'uint8', 'uint8')",0.0,"[2825, 13, 389]",3227
15,la_palma-volcano_00000691,pre-event,3,1024,1024,"('uint8', 'uint8', 'uint8')",0.0,"[4974, 3677, 2477]",11128
16,la_palma-volcano_00000691,post-event,1,1024,1024,"('uint8',)",0.0,[11455],11455
18,libya-flood_00000017,pre-event,3,1024,1024,"('uint8', 'uint8', 'uint8')",0.0,"[255354, 255354, 255354]",766062
19,libya-flood_00000017,post-event,1,1024,1024,"('uint8',)",0.0,[255347],255347


## Validità congiunta delle modalità

Confrontiamo due criteri di validità dell'immagine
ottica, combinandoli con le maschere di validità
del SAR e delle annotazioni.

L'obiettivo è quantificare i pixel effettivamente
utilizzabili e definire una politica di gestione
dei NoData prima dell'addestramento.

In [17]:
from bright_reproduction.campione import (
    statistiche_validita_congiunta
)

risultati_validita = {}

for split, identificativi in campione_operativo.items():

    tabella = statistiche_validita_congiunta(
        CARTELLA_BASELINE / split,
        identificativi
    )

    risultati_validita[split] = tabella

    print(f"\n{split.upper()}")

    display(tabella.round(2))

    print(
        "Percentuale valida rigorosa:",
        round(
            100 * tabella["validi_rigoroso"].sum()
            / tabella["pixel_totali"].sum(), 2
        )
    )

    print(
        "Percentuale valida alternativa:",
        round(
            100 * tabella["validi_alternativo"].sum()
            / tabella["pixel_totali"].sum(), 2
        )
    )


TRAIN


,pixel_totali,validi_rigoroso,validi_alternativo,percentuale_rigoroso,percentuale_alternativo
id,,,,,
bata-explosion_00000096,1048576,58353,58353,5.56,5.56
bata-explosion_00000016,1048576,990632,991916,94.47,94.60
beirut-explosion_00000005,1048576,1008303,1048450,96.16,99.99
beirut-explosion_00000110,1048576,420411,437447,40.09,41.72
congo-volcano_00000041,1048576,1002456,1002474,95.60,95.60
congo-volcano_00000037,1048576,1006714,1006782,96.01,96.01
haiti-earthquake_00000015,1048576,1046436,1048576,99.80,100.00
haiti-earthquake_00000009,1048576,925757,928184,88.29,88.52
hawaii-wildfire_00000054,1048576,1048576,1048576,100.00,100.00


Percentuale valida rigorosa: 83.08
Percentuale valida alternativa: 84.14

VAL


,pixel_totali,validi_rigoroso,validi_alternativo,percentuale_rigoroso,percentuale_alternativo
id,,,,,
bata-explosion_00000012,1048576,74405,74405,7.10,7.10
beirut-explosion_00000047,1048576,1005475,1048521,95.89,99.99
congo-volcano_00000014,1048576,1012876,1012878,96.60,96.60
haiti-earthquake_00000059,1048576,1045474,1048576,99.70,100.00
hawaii-wildfire_00000002,1048576,1048576,1048576,100.00,100.00
la_palma-volcano_00000691,1048576,1031143,1036506,98.34,98.85
libya-flood_00000017,1048576,793222,793222,75.65,75.65
morocco-earthquake_00000448,1048576,1003324,1009574,95.68,96.28
turkey-earthquake_00000845,1048576,1048345,1048576,99.98,100.00


Percentuale valida rigorosa: 85.44
Percentuale valida alternativa: 86.05


## Preparazione delle patch multimodali

Adottiamo la normalizzazione ImageNet utilizzata
nell'implementazione originale di BRIGHT.

Introduciamo la lettura per finestre dei GeoTIFF
e la gestione congiunta della validità dei pixel.

I pixel non validi vengono esclusi dalla loss
mediante ignore_index=255.

In [18]:
from bright_reproduction.data import (
    carica_patch_multimodale
)

patch_test = carica_patch_multimodale(
    CARTELLA_BASELINE / "train",
    "bata-explosion_00000016",
    x=448,
    y=448,
    dimensione=128
)

print("ID:", patch_test["id"])
print("Input:", patch_test["input"].shape)
print("Target:", patch_test["target"].shape)

print("Tipo input:", patch_test["input"].dtype)
print("Tipo target:", patch_test["target"].dtype)

print("Classi:", torch.unique(patch_test["target"]).tolist())

print("Pixel validi:", patch_test["validita"].sum().item())

print(
    "Pixel ignorati:",
    (patch_test["target"] == 255).sum().item()
)

print(
    "Input finito:",
    torch.isfinite(patch_test["input"]).all().item()
)

assert torch.equal(
    patch_test["target"] != 255,
    patch_test["validita"]
)

ID: bata-explosion_00000016
Input: torch.Size([6, 128, 128])
Target: torch.Size([128, 128])
Tipo input: torch.float32
Tipo target: torch.int64
Classi: [0, 1, 255]
Pixel validi: 15647
Pixel ignorati: 737
Input finito: True


## Dataset per l'addestramento e la validation

Suddividiamo le immagini del campione operativo
in patch non sovrapposte da 128 × 128 pixel.

Nel training escludiamo le patch completamente
prive di pixel validi.

Nella validation manteniamo la copertura completa,
escludendo i pixel non validi dalle metriche.

In [19]:
from bright_reproduction.data import BrightPatchDataset

DIM_PATCH_BASELINE = 128

dataset_train = BrightPatchDataset(
    CARTELLA_BASELINE / "train",
    campione_operativo["train"],
    dimensione=DIM_PATCH_BASELINE,
    training=True
)

dataset_val = BrightPatchDataset(
    CARTELLA_BASELINE / "val",
    campione_operativo["val"],
    dimensione=DIM_PATCH_BASELINE,
    training=False
)

print("Patch candidate training:", 18 * 64)
print("Patch training utilizzabili:", len(dataset_train))
print("Patch escluse:", 18 * 64 - len(dataset_train))

print("Patch validation:", len(dataset_val))

assert 0 < len(dataset_train) <= 1152
assert len(dataset_val) == 576

Patch candidate training: 1152
Patch training utilizzabili: 1017
Patch escluse: 135
Patch validation: 576


In [20]:
esempio = dataset_train[0]

print("ID:", esempio["id"])
print("Coordinate:", esempio["x"], esempio["y"])
print("Input:", esempio["input"].shape)
print("Target:", esempio["target"].shape)
print("Classi:", torch.unique(esempio["target"]).tolist())
print("Pixel validi:", esempio["validita"].sum().item())

assert esempio["input"].shape == (6, 128, 128)
assert esempio["target"].shape == (128, 128)
assert esempio["validita"].any().item()
assert torch.isfinite(esempio["input"]).all().item()

print("Test del Dataset superato.")

ID: bata-explosion_00000096
Coordinate: 0 0
Input: torch.Size([6, 128, 128])
Target: torch.Size([128, 128])
Classi: [0, 255]
Pixel validi: 16039
Test del Dataset superato.


## Distribuzione delle classi nelle patch di training

Analizziamo la distribuzione delle classi dopo
l'applicazione della maschera di validità congiunta.

Contiamo inoltre le patch contenenti edifici
integri, danneggiati e distrutti, per definire
una strategia di campionamento che tenga conto
dello sbilanciamento delle classi.

L'analisi finalizzata al campionamento utilizza
esclusivamente i dati di training.

In [21]:
from bright_reproduction.data import statistiche_patch

statistiche_patch_train = statistiche_patch(
    dataset_train
)

display(statistiche_patch_train.head())

,indice,id,x,y,pixel_validi,pixel_ignorati,sfondo,integro,danneggiato,distrutto
0,0,bata-explosion_00000096,0,0,16039,345,16039,0,0,0
1,1,bata-explosion_00000096,128,0,14684,1700,14684,0,0,0
2,2,bata-explosion_00000096,256,0,2115,14269,2115,0,0,0
3,3,bata-explosion_00000096,0,128,15593,791,15593,0,0,0
4,4,bata-explosion_00000096,128,128,3653,12731,3653,0,0,0


In [22]:
colonne_classi = [
    "sfondo",
    "integro",
    "danneggiato",
    "distrutto"
]

conteggi = statistiche_patch_train[
    colonne_classi
].sum()

riepilogo_patch_train = pd.DataFrame({
    "pixel_validi": conteggi,

    "percentuale": (
        100 * conteggi / conteggi.sum()
    ),

    "patch_con_classe": (
        statistiche_patch_train[colonne_classi] > 0
    ).sum()
})

display(riepilogo_patch_train.round(2))

patch_solo_sfondo = (
    statistiche_patch_train[
        ["integro", "danneggiato", "distrutto"]
    ].sum(axis=1) == 0
).sum()

print("Patch totali:", len(statistiche_patch_train))
print("Patch solo sfondo:", patch_solo_sfondo)

print(
    "Patch contenenti edifici:",
    len(statistiche_patch_train) - patch_solo_sfondo
)

assert (
    conteggi.sum()
    == risultati_validita["train"]["validi_rigoroso"].sum()
)

print("Coerenza dei conteggi verificata.")

,pixel_validi,percentuale,patch_con_classe
sfondo,13560670,86.48,1017
integro,1701722,10.85,434
danneggiato,337772,2.15,68
distrutto,80743,0.51,44


Patch totali: 1017
Patch solo sfondo: 513
Patch contenenti edifici: 504
Coerenza dei conteggi verificata.


## Salvataggio della preparazione sperimentale

Salviamo gli identificativi del campione operativo,
la configurazione delle patch e le statistiche
necessarie per verificare la coerenza dei successivi
esperimenti.

I notebook di addestramento potranno ricostruire
i Dataset senza ripetere tutte le analisi precedenti.

In [23]:
import json

# Save the selected sample in its original order
for split, identificativi in campione_operativo.items():

    percorso = (
        CARTELLA_SPLIT_OPERATIVI
        / f"campione_{split}.txt"
    )

    percorso.write_text(
        "\n".join(identificativi) + "\n",
        encoding="utf-8"
    )

configurazione_dati = {
    "patch_size": DIM_PATCH_BASELINE,
    "ignore_index": 255,
    "normalizzazione": "ImageNet",
    "validita_ottico": "rigorosa",
    "tolleranza_griglie": TOLLERANZA_GRIGLIE,
    "seed_campione": 42,
    "patch_train": len(dataset_train),
    "patch_val": len(dataset_val),
    "pixel_validi_train": int(conteggi.sum()),
    "pixel_validi_val": int(
        risultati_validita["val"]["validi_rigoroso"].sum()
    )
}

percorso_configurazione = (
    CARTELLA_SPLIT_OPERATIVI
    / "configurazione_dati.json"
)

percorso_configurazione.write_text(
    json.dumps(configurazione_dati, indent=4) + "\n",
    encoding="utf-8"
)

print("Preparazione sperimentale salvata.")

display(pd.Series(configurazione_dati))

Preparazione sperimentale salvata.


patch_size                 128
ignore_index               255
normalizzazione       ImageNet
validita_ottico       rigorosa
tolleranza_griglie        0.05
seed_campione               42
patch_train               1017
patch_val                  576
pixel_validi_train    15680907
pixel_validi_val       8062840
dtype: object